# R0 — 평가 세트 확장과 근거 문서 중복(오염) 점검

## 왜 하나

다음 모델링(R2 순차 판정 규칙, R3 오경보 보장 보정, R1 발생 시점 학습)을 시작하기 전에 **평가 자체를 고쳐야** 한다. 문제는 두 가지다.

1. **검정력 부족**: 지금 dev의 환각 응답은 154개뿐이다. 엄격 재현율의 95% 구간이 ±0.08이고, 시드 간 편차도 0.015~0.03이다(E4-lite). 0.05 수준의 개선은 측정할 수 없다.
2. **근거 문서 중복**: RAGTruth는 근거 문서 하나마다 6개 모델의 응답이 붙어 있다. 기존 분할은 **응답** 단위로 나눴기 때문에, dev 응답의 **62%가 학습(gold-large)에 쓰인 근거 문서와 같은 문서**를 쓴다. 판정기는 같은 문서에 대한 다른 모델의 응답과 정답을 이미 학습했다. dev 수치가 낙관적일 수 있다.

## 무엇을 하나

1. train 분할에서 **학습·검증·dev 어느 것과도 근거 문서가 겹치지 않는** 응답만 모은다(약 3,200개, 근거 536개, 환각 응답 약 1,500개).
   근거 문서 단위로, 과제별로 반씩 나눠 **cal-L(보정·규칙 학습용)** 과 **dev-L(평가용)** 을 만든다. 새 시드 `random.Random(2027)`.
2. 기존 판정기 5개(student_vark = B, student_prefix = E2, E4-lite 시드 모델 S42·S7·S2026)로 네 세트(dev, 검증, cal-L, dev-L)의 **모든 판정 지점(2토큰 간격 + 문장 끝) 점수**를 계산해 드라이브에 저장한다.
   이 점수 궤적이 R2·R3의 재료다. 이후 실험은 GPU 없이 이 파일로 돈다.
3. 기존 dev 안에서 **학습에 본 근거 문서(seen)** 와 **처음 보는 문서(unseen)** 의 성능을 따로 재서 오염 효과를 직접 측정한다.
4. dev-L에서 모든 판정기를 E1 프로토콜(엄격 기준, ARL₀=100, 서비스 10%) 그대로 다시 평가한다. 임계값은 cal-L로 보정한다.

## 사전 정의 기준 (결과 보기 전, SSOT §6에 고정)

- **R0-a 분할 무결성**: cal-L·dev-L의 근거 문서가 학습(gold-large)·검증·dev와 겹치지 않고, cal-L과 dev-L끼리도 겹치지 않는다 (`assert`).
- **R0-b 재현**: 기존 dev에서 B·E2 점수가 E4-lite 기록과 일치한다. 완성 문장 AUROC 0.859 / 0.881 (±0.003), ARL₀=100 엄격 재현율 B1 0.474 / E2 s=4 0.494 (±0.013, 응답 2개). 어긋나면 이후 수치는 해석하지 않는다.
- **R0-c 오염 판정**: 기존 dev에서 B의 완성 문장 AUROC 차이(seen − unseen, **과제별 차이의 평균**)가 **0.02 초과이고 부트스트랩 95% 구간 하한이 0보다 크면** "기존 dev 수치는 문서 중복으로 부풀려짐"으로 기록한다. 그렇지 않으면 "오염 영향은 측정 범위 안에서 작음"으로 기록한다.
- **결정(결과와 무관하게 지금 고정)**: 이후 모든 주장의 주 평가 세트는 **dev-L**, 임계값 보정과 규칙 학습은 **cal-L**로 한다. 기존 dev 수치는 참고로만 쓴다.
- **보고(판정 없음)**: dev-L에서 모든 판정기의 재현율·지연·AUROC, 쌍체 부트스트랩 구간, 기존 dev 대비 구간 폭.

> 런타임 **GPU(T4)**. 판정 질의 약 35만~40만 개 × 모델 5개 → **약 1시간**. 점수는 2만 개 단위 조각으로 드라이브 `selfproject/r0/scores/`에 저장되므로,
> 끊겨도 다시 실행하면 끝난 조각은 건너뛴다. 드라이브에 `student_vark`, `student_prefix`, `student_prefix_seeds/seed*_ep2`가 있어야 한다(시드 모델은 없으면 건너뜀).


## 1. 환경 준비

In [ ]:
!pip -q install pysbd sentence-transformers tiktoken
!git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || echo 'RAGTruth already cloned'

import json, time, random, os, gc, copy
import numpy as np, pandas as pd, torch, pysbd, tiktoken
from sklearn.metrics import roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

USE_DRIVE = True
OUT_DIR = '/content/drive/MyDrive/selfproject' if USE_DRIVE else '.'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
STUDENT_DIR = os.path.join(OUT_DIR, 'student_vark')
E2_DIR = os.path.join(OUT_DIR, 'student_prefix')
SEED_ROOT = os.path.join(OUT_DIR, 'student_prefix_seeds')
R0_DIR = os.path.join(OUT_DIR, 'r0')
os.makedirs(R0_DIR, exist_ok=True)
assert os.path.isdir(STUDENT_DIR), f'{STUDENT_DIR} 없음'
print('device', DEVICE)

## 2. 데이터 — 기존 dev·검증·학습 분할 (E1·E2 코드 그대로)

In [ ]:
BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources   = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{BASE}/source_info.jsonl')}
for r in responses:
    r['task_type'] = sources[r['source_id']]['task_type']

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))

TASKS = ['QA', 'Summary', 'Data2txt']
train_pool = [r for r in responses if r['split'] == 'train']          # test 분할은 사용하지 않음
random.seed(SEED)
dev = []
for tt in TASKS:
    dev += random.sample([r for r in train_pool if r['task_type'] == tt], 120)
dev_ids = {r['id'] for r in dev}
rest = [r for r in train_pool if r['id'] not in dev_ids]
rng = random.Random(11)
teacher_resp = []
for tt in TASKS:
    teacher_resp += rng.sample([r for r in rest if r['task_type'] == tt], 200)
rng.shuffle(teacher_resp)
val = teacher_resp[:int(len(teacher_resp) * 0.15)]                   # student 학습에 쓰지 않은 검증 분할
assert not (dev_ids & {r['id'] for r in val})
print(f'dev {len(dev)} 응답 | 검증 {len(val)} 응답')

In [ ]:
# 학습 응답: S2/S2-b의 gold-large와 같은 구성 (teacher 학습 510 + 추가 2,490). dev·검증과 겹치지 않음을 확인한다.
teacher_train = teacher_resp[len(val):]
used = dev_ids | {r['id'] for r in teacher_resp}
gl_extra = rng.sample([r for r in rest if r['id'] not in used], 3000 - len(teacher_train))
train = teacher_train + gl_extra
tr_ids = {r['id'] for r in train}
assert not (tr_ids & dev_ids) and not (tr_ids & {r['id'] for r in val}), '학습과 dev/검증 겹침'
print(f'학습 {len(train)} 응답 (dev·검증과 겹침 없음)')

## 3. 새 분할 — 근거 문서가 겹치지 않는 cal-L / dev-L

근거 문서(source_id) 단위로 나눈다. 같은 문서의 응답 6개는 항상 같은 쪽에 들어간다.

In [ ]:
S = lambda rs: {r['source_id'] for r in rs}
TRAIN_SRC = S(train)
used_src = TRAIN_SRC | S(val) | S(dev) | S(teacher_resp)
cand_src = sorted(S(train_pool) - used_src)
r0rng = random.Random(2027)
cal_src, devL_src = set(), set()
for tt in TASKS:
    ss = [s for s in cand_src if sources[s]['task_type'] == tt]
    r0rng.shuffle(ss)
    h = len(ss) // 2
    cal_src |= set(ss[:h]); devL_src |= set(ss[h:])
calL = [r for r in train_pool if r['source_id'] in cal_src]
devL = [r for r in train_pool if r['source_id'] in devL_src]

# R0-a 분할 무결성
assert not (S(calL) & used_src) and not (S(devL) & used_src), '새 분할이 기존 분할과 근거 문서를 공유'
assert not (S(calL) & S(devL)), 'cal-L과 dev-L이 근거 문서를 공유'
assert all(r['split'] == 'train' for r in calL + devL), 'test 분할 사용 금지'
ids = lambda rs: {str(r['id']) for r in rs}
assert not (ids(calL) & ids(devL)) and not ((ids(calL) | ids(devL)) & (ids(train) | ids(val) | ids(dev)))
print('R0-a 분할 무결성: 충족')

def desc(name, rs):
    n_h = sum(1 for r in rs if r.get('labels'))
    tt = pd.Series([r['task_type'] for r in rs]).value_counts().to_dict()
    print(f'{name:6} 응답 {len(rs):5} | 근거 {len(S(rs)):4} | 환각 응답 {n_h:5} | 과제 {tt}')
for name, rs in [('dev', dev), ('검증', val), ('cal-L', calL), ('dev-L', devL)]:
    desc(name, rs)
seen = {str(r['id']) for r in dev if r['source_id'] in TRAIN_SRC}
print(f'기존 dev 중 학습 근거 문서를 공유하는 응답: {len(seen)}/{len(dev)} ({len(seen)/len(dev):.0%})')

json.dump({'seed': 2027, 'cal_L_ids': sorted(ids(calL)), 'dev_L_ids': sorted(ids(devL)),
           'cal_L_sources': sorted(cal_src), 'dev_L_sources': sorted(devL_src), 'dev_seen_ids': sorted(seen)},
          open(os.path.join(R0_DIR, 'r0_splits.json'), 'w'))

## 4. 토큰 스트림·문장·발생 위치 (E1 코드 그대로)

In [ ]:
ENC = tiktoken.get_encoding('cl100k_base')
seg = pysbd.Segmenter(language='en', clean=False, char_span=True)

def tokenize_offsets(text):
    ids = ENC.encode(text)
    dec, starts = ENC.decode_with_offsets(ids)
    if dec != text:                       # 드문 경우: 멀티바이트 문자 경계 — 누적 디코드로 대체
        starts, pos = [], 0
        for i in range(len(ids)):
            starts.append(pos); pos = len(ENC.decode(ids[:i + 1]))
    ends = starts[1:] + [len(text)]
    return starts, ends

def prep(r):
    text = r['response']
    ts, te = tokenize_offsets(text)
    # 토큰의 '내용 시작' 위치: cl100k 토큰은 앞 공백을 포함하므로(" The"), 공백을 건너뛴 위치로 문장·환각 구간에 배정한다.
    # (v1에서는 공백 위치로 배정해 문장 경계 토큰이 앞 문장에 붙는 오류가 발생 — 발생 위치의 18%)
    cs = [a + (len(text[a:b]) - len(text[a:b].lstrip())) for a, b in zip(ts, te)]
    cs = [c if c < b else a for c, a, b in zip(cs, ts, te)]          # 공백뿐인 토큰
    spans = [(l['start'], l['end']) for l in (r.get('labels') or [])]
    bad = [j for j in range(len(ts)) if any(min(te[j], b) > max(cs[j], a) for a, b in spans)]
    sents = []
    for sp in seg.segment(text):
        if not sp.sent.strip(): continue
        a, b = sp.start, sp.end
        tok = [j for j in range(len(ts)) if a <= cs[j] < b]
        if not tok: continue
        y = int(any(min(b, e) > max(a, s) for s, e in spans))
        sents.append({'a': a, 'b': b, 't0': tok[0], 't1': tok[-1], 'y': y})
    sent_of = {t: sn['t0'] for sn in sents for t in range(sn['t0'], sn['t1'] + 1)}
    return {'id': str(r['id']), 'task_type': r['task_type'], 'gen_model': r['model'],
            'ref_key': r['source_id'], 'text': text, 'ts': ts, 'te': te, 'n_tok': len(ts),
            'onset': bad[0] if bad else None, 'sents': sents, 'sent_of': sent_of,
            'ends': {sn['t1'] for sn in sents}}

REFS = {}
def to_items(rs):
    out = []
    for r in rs:
        REFS.setdefault(r['source_id'], build_reference(sources[r['source_id']]))
        out.append(prep(r))
    return out

dev_items, val_items = to_items(dev), to_items(val)
for name, it in [('dev', dev_items), ('검증', val_items)]:
    n_h = sum(x['onset'] is not None for x in it)
    print(f'{name}: 응답 {len(it)} (환각 {n_h}, 없음 {len(it)-n_h}) | 토큰 {sum(x["n_tok"] for x in it):,} | '
          f'환각 없는 응답 토큰 {sum(x["n_tok"] for x in it if x["onset"] is None):,}')

In [ ]:
calL_items, devL_items = to_items(calL), to_items(devL)
by_all = {str(r['id']): r for r in dev + val + calL + devL}
ALL = {'dev': dev_items, 'val': val_items, 'calL': calL_items, 'devL': devL_items}
for name, its in ALL.items():
    for x in its:
        x['labels'] = by_all[x['id']].get('labels') or []
        text = x['text']
        x['cs'] = [a + (len(text[a:b]) - len(text[a:b].lstrip())) if text[a:b].strip() else a
                   for a, b in zip(x['ts'], x['te'])]
        x['seen'] = x['id'] in seen
    n_h = sum(x['onset'] is not None for x in its)
    print(f'{name:5} 응답 {len(its):5} (환각 {n_h:5}) | 토큰 {sum(x["n_tok"] for x in its):,}')

## 5. 판정 지점 (E1과 같음: 문장 안 2토큰 간격 + 문장 끝)

In [ ]:
def points(item, s):
    pts = []
    for sn in item['sents']:
        for t in range(sn['t0'], sn['t1'] + 1):
            k = t - sn['t0'] + 1
            if (s is not None and k % s == 0) or t == sn['t1']:
                pts.append((t, item['text'][sn['a']:min(item['te'][t], sn['b'])].strip()))
    return pts


QUERY_ID, queries = {}, []
def qid(key, text):
    if (key, text) not in QUERY_ID:
        QUERY_ID[(key, text)] = len(queries); queries.append((key, text))
    return QUERY_ID[(key, text)]


def point_labels(items):
    ids, ys = [], []
    for x in items:
        spans = [(l['start'], l['end']) for l in x['labels']]
        for t, q in x['pts2']:
            a = x['sent_of'][t]
            ys.append(int(any(min(x['te'][u], b) > max(x['cs'][u], s0) for u in range(a, t + 1) for s0, b in spans)))
            ids.append(q)
    return np.array(ids), np.array(ys)


for its in ALL.values():
    for x in its:
        x['pts2'] = [(t, qid(x['ref_key'], txt)) for t, txt in points(x, 2) if txt]
Q_KEYS, Q_TEXTS = [q[0] for q in queries], [q[1] for q in queries]
print(f'판정 질의 {len(queries):,}개')

## 6. 근거 검색과 판정 — 모델 5개를 한 번에, 조각마다 드라이브에 저장

근거 검색(top-8 조각)은 질의마다 한 번만 하고, 같은 입력을 모든 판정기에 넣는다.
조각 파일에 없는 모델만 새로 계산하므로, 나중에 시드 모델이 생겨도 이어서 채울 수 있다.

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

retriever = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)
K, MAX_LEN, CHUNK_WORDS, BS = 8, 512, 40, 32

chunk_cache = {}
def get_chunks(key):
    if key not in chunk_cache:
        w = REFS[key].split()
        ch = [' '.join(w[i:i + CHUNK_WORDS]) for i in range(0, max(len(w), 1), CHUNK_WORDS)] or [REFS[key]]
        chunk_cache[key] = (ch, retriever.encode(ch, convert_to_tensor=True, normalize_embeddings=True,
                                                 batch_size=128, show_progress_bar=False))
    return chunk_cache[key]

def contexts(keys, texts, bs=256):
    # 텍스트마다 근거 top-K 조각을 원문 순서로 이어 붙인 문자열
    out = []
    for i in range(0, len(texts), bs):
        embs = retriever.encode(texts[i:i + bs], convert_to_tensor=True, normalize_embeddings=True,
                                batch_size=bs, show_progress_bar=False)
        for key, e in zip(keys[i:i + bs], embs):
            ch, ce = get_chunks(key)
            top = torch.topk(ce @ e, min(K, ce.shape[0])).indices.tolist()
            out.append(' ... '.join(ch[t] for t in sorted(top)))
    return out


import hashlib, math
MODEL_DIRS = {'B': STUDENT_DIR, 'E2': E2_DIR}
MODEL_DIRS.update({f'S{s}': os.path.join(SEED_ROOT, f'seed{s}_ep2') for s in [42, 7, 2026]})
missing = [k for k, v in MODEL_DIRS.items() if not os.path.exists(os.path.join(v, 'config.json'))]
MODEL_DIRS = {k: v for k, v in MODEL_DIRS.items() if k not in missing}
assert 'B' in MODEL_DIRS and 'E2' in MODEL_DIRS, 'student_vark·student_prefix가 드라이브에 있어야 한다'
print('판정기:', list(MODEL_DIRS), '| 없어서 건너뜀:', missing)

TOKS, MODELS = {}, {}
for k, v in MODEL_DIRS.items():
    TOKS[k] = AutoTokenizer.from_pretrained(v)
    MODELS[k] = AutoModelForSequenceClassification.from_pretrained(v).to(DEVICE).eval()
same_tok = all(TOKS[k].get_vocab() == TOKS['B'].get_vocab() for k in TOKS)
print('토크나이저 공유:', same_tok)

SC_DIR = os.path.join(R0_DIR, 'scores'); os.makedirs(SC_DIR, exist_ok=True)
sig = hashlib.md5('\n'.join(f'{k}\t{t}' for k, t in queries).encode()).hexdigest()
meta_p = os.path.join(SC_DIR, 'meta.json')
if os.path.exists(meta_p):
    assert json.load(open(meta_p))['sig'] == sig, '저장된 조각과 질의 목록이 다름 — scores/ 폴더를 비우고 다시 실행'
json.dump({'sig': sig, 'n_queries': len(queries)}, open(meta_p, 'w'))

SHARD, BSZ = 20000, 128
order = sorted(range(len(queries)), key=lambda i: (len(Q_TEXTS[i]), i))
n_sh = math.ceil(len(order) / SHARD)

@torch.no_grad()
def run_batch(idx, names):
    ctx = contexts([Q_KEYS[j] for j in idx], [Q_TEXTS[j] for j in idx])
    txt = [Q_TEXTS[j] for j in idx]
    out, enc = {}, None
    for n in names:
        if enc is None or not same_tok:
            enc = TOKS[n](ctx, txt, truncation='only_first', max_length=MAX_LEN,
                          padding=True, return_tensors='pt').to(DEVICE)
        with torch.autocast(DEVICE, enabled=(DEVICE == 'cuda')):          # E2와 같은 정밀도 (재현 확인용)
            out[n] = MODELS[n](**enc).logits.squeeze(-1).float().cpu().numpy()
    return out

t0 = time.perf_counter(); done_q = 0
for sh in range(n_sh):
    p = os.path.join(SC_DIR, f'shard{sh:03d}.npz')
    idx_all = np.array(order[sh * SHARD:(sh + 1) * SHARD])
    have = dict(np.load(p)) if os.path.exists(p) else {}
    todo = [n for n in MODEL_DIRS if n not in have]
    if not todo:
        continue
    res = {n: np.zeros(len(idx_all), dtype=np.float32) for n in todo}
    for i in range(0, len(idx_all), BSZ):
        b = idx_all[i:i + BSZ]
        for n, v in run_batch(b.tolist(), todo).items():
            res[n][i:i + len(b)] = v
    have.update(res); have['idx'] = idx_all
    np.savez(p, **have)
    done_q += len(idx_all)
    el = time.perf_counter() - t0
    print(f'조각 {sh + 1}/{n_sh} 저장 ({", ".join(todo)}) | {el/60:.1f}분 경과, '
          f'남은 예상 {el / done_q * (len(order) - (sh + 1) * SHARD) / 60:.0f}분')

SC = {n: np.full(len(queries), np.nan, dtype=np.float32) for n in MODEL_DIRS}
for sh in range(n_sh):
    z = np.load(os.path.join(SC_DIR, f'shard{sh:03d}.npz'))
    for n in MODEL_DIRS:
        SC[n][z['idx']] = z[n]
assert all(not np.isnan(v).any() for v in SC.values()), '빠진 점수가 있음'
print(f'점수 준비 완료: {len(queries):,}개 × {len(SC)}개 모델')

## 7. 평가 함수 (E1 코드 그대로)

점수기 이름은 `모델|판정 시점` 형식이다. 예: `B|end`(문장 끝), `E2|4`(4토큰마다 앞부분 + 문장 끝).

In [ ]:
def series(x, scorer):
    m, k = scorer.split('|')
    S_ = SC[m]
    if k == 'end':
        return [(t, S_[q]) for t, q in x['pts2'] if t in x['ends']]
    s = int(k)
    return [(t, S_[q]) for t, q in x['pts2'] if t in x['ends'] or (t - x['sent_of'][t] + 1) % s == 0]

In [ ]:
def calibrate(items, scorer, arl0):
    clean = [x for x in items if x['onset'] is None]
    n_tok = sum(x['n_tok'] for x in clean)
    sc = np.sort(np.concatenate([[s for _, s in series(x, scorer)] for x in clean]))[::-1]
    n_alarm = max(1, int(n_tok / arl0))
    return float(sc[n_alarm - 1]) + 1e-6 if n_alarm <= len(sc) else float(sc[-1])

def evaluate(items, scorer, thr):
    # 엄격 기준(주 지표, 선행 연구와 같은 정의): 응답의 **첫 경보**가 발생 θ 이후여야 탐지로 인정.
    #   θ 전에 경보가 울리면 그 응답은 오경보로 끝난 것(미탐지)으로 본다.
    # 관대 기준(참고): θ 전 경보는 무시하고 θ 이후 첫 경보를 탐지로 인정 — v1의 주 지표였으나 재현율이 부풀려짐.
    clean = [x for x in items if x['onset'] is None]
    hall = [x for x in items if x['onset'] is not None]
    alarms = sum(sum(s >= thr for _, s in series(x, scorer)) for x in clean)
    arl = sum(x['n_tok'] for x in clean) / max(alarms, 1)
    clean_flag = np.mean([any(s >= thr for _, s in series(x, scorer)) for x in clean])
    st_det, st_d, st_ch, cedd, le_det, pre = 0, [], [], [], 0, 0
    for x in hall:
        ser = series(x, scorer)
        first = next((t for t, s in ser if s >= thr), None)
        le_hit = next((t for t, s in ser if t >= x['onset'] and s >= thr), None)
        le_det += le_hit is not None
        pre += first is not None and first < x['onset']
        if first is not None and first >= x['onset']:
            st_det += 1; st_d.append(first - x['onset']); cedd.append(first - x['onset'])
            st_ch.append(x['te'][first] - x['ts'][x['onset']])
        else:
            cedd.append(x['n_tok'] - x['onset'])
    return {'realized_ARL0_dev': arl, 'clean_resp_flagged': clean_flag,
            'recall': st_det / len(hall), 'delay_tok_mean': np.mean(st_d) if st_d else np.nan,
            'delay_tok_median': np.median(st_d) if st_d else np.nan,
            'delay_chars_median': np.median(st_ch) if st_ch else np.nan,
            'censored_EDD_tok': np.mean(cedd), 'pre_onset_alarm_rate': pre / len(hall),
            'recall_lenient': le_det / len(hall),
            'calls_per_resp': np.mean([len(series(x, scorer)) for x in items])}

def calibrate_flag(items, scorer, rate):
    # 서비스 기준 보정: 환각 없는 응답 중 표시되는 비율이 rate 이하가 되는 임계값 (응답별 최대 점수의 분위수)
    mx = np.array([max([s for _, s in series(x, scorer)] or [-1e9]) for x in items if x['onset'] is None])
    return float(np.quantile(mx, 1 - rate)) + 1e-6

In [ ]:
def sent_auc(items, m):
    ys, ss = [], []
    for x in items:
        d = dict(x['pts2'])
        for sn in x['sents']:
            if sn['t1'] in d:
                ys.append(sn['y']); ss.append(SC[m][d[sn['t1']]])
    return roc_auc_score(ys, ss) if len(set(ys)) == 2 else np.nan

def prefix_auc(items, m):
    q, y = point_labels(items)
    return roc_auc_score(y, SC[m][q])

## 8. R0-b 재현 — 기존 dev에서 E4-lite 기록과 같은가

In [ ]:
REF = {'auc_B': 0.8590, 'auc_E2': 0.8807, 'rec_B1': 0.4740, 'rec_E2s4': 0.4935}
got = {'auc_B': sent_auc(dev_items, 'B'), 'auc_E2': sent_auc(dev_items, 'E2'),
       'rec_B1': evaluate(dev_items, 'B|end', calibrate(val_items, 'B|end', 100))['recall'],
       'rec_E2s4': evaluate(dev_items, 'E2|4', calibrate(val_items, 'E2|4', 100))['recall']}
TOL = {'auc_B': 0.003, 'auc_E2': 0.003, 'rec_B1': 0.013, 'rec_E2s4': 0.013}
R0B = all(abs(got[k] - REF[k]) <= TOL[k] for k in REF)
for k in REF:
    print(f'{k:9} 기록 {REF[k]:.4f} | 지금 {got[k]:.4f} | {"일치" if abs(got[k]-REF[k]) <= TOL[k] else "불일치"}')
print('R0-b 재현:', '충족' if R0B else '미달 — 이후 수치를 해석하지 말고 원인부터 확인')

## 9. R0-c 오염 — 기존 dev의 seen(학습 근거 문서 공유) vs unseen

과제 구성이 두 집단에서 다를 수 있으므로, **과제별 AUROC 차이의 평균**을 주 지표로 쓴다. 구간은 응답 단위 부트스트랩(1,000회).

In [ ]:
def strat_diff(items, m, rng=None):
    ds = []
    for tt in TASKS:
        its = [x for x in items if x['task_type'] == tt]
        if rng is not None:
            its = [its[i] for i in rng.integers(0, len(its), len(its))]
        a = sent_auc([x for x in its if x['seen']], m)
        b = sent_auc([x for x in its if not x['seen']], m)
        ds.append(a - b)
    return float(np.nanmean(ds))

brng = np.random.default_rng(0)
rows = []
for m in SC:
    d0 = strat_diff(dev_items, m)
    bs = [strat_diff(dev_items, m, brng) for _ in range(1000)]
    lo, hi = np.nanpercentile(bs, [2.5, 97.5])
    rows.append({'model': m, 'auc_seen': sent_auc([x for x in dev_items if x['seen']], m),
                 'auc_unseen': sent_auc([x for x in dev_items if not x['seen']], m),
                 'strat_diff': d0, 'ci_lo': lo, 'ci_hi': hi,
                 'auc_dev_all': sent_auc(dev_items, m), 'auc_devL': sent_auc(devL_items, m)})
contam = pd.DataFrame(rows).set_index('model')
pd.set_option('display.width', 250)
print(contam.round(3).to_string())
b = contam.loc['B']
R0C = b.strat_diff > 0.02 and b.ci_lo > 0
print(f"\nR0-c (B, 과제별 차이 평균 {b.strat_diff:+.3f} [{b.ci_lo:+.3f}, {b.ci_hi:+.3f}]): "
      + ('오염 효과 있음 — 기존 dev 수치는 부풀려짐' if R0C else '오염 효과는 측정 범위 안에서 작음'))

# 과제별: 기존 dev(seen / unseen) vs dev-L
pt = []
for m in ['B', 'E2']:
    for tt in TASKS:
        d_ = [x for x in dev_items if x['task_type'] == tt]
        pt.append({'model': m, 'task': tt, 'dev_seen': sent_auc([x for x in d_ if x['seen']], m),
                   'dev_unseen': sent_auc([x for x in d_ if not x['seen']], m),
                   'devL': sent_auc([x for x in devL_items if x['task_type'] == tt], m)})
contam_task = pd.DataFrame(pt)
print('\n과제별 완성 문장 AUROC'); print(contam_task.round(3).to_string(index=False))

# 엄격 재현율도 seen/unseen으로 (임계값은 기존 검증으로 보정, E2와 같음)
for sc in ['B|end', 'E2|4']:
    thr = calibrate(val_items, sc, 100)
    for flag in [True, False]:
        its = [x for x in dev_items if x['seen'] == flag]
        r = evaluate(its, sc, thr)
        n_h = sum(x['onset'] is not None for x in its)
        print(f"{sc:6} {'seen  ' if flag else 'unseen'} 환각 {n_h:3}개 | 엄격 재현율 {r['recall']:.3f} | 멀쩡한 응답 표시 {r['clean_resp_flagged']:.1%}")

## 10. dev-L 평가 — cal-L로 보정, E1 프로토콜 그대로

In [ ]:
res = []
for m in SC:
    auc_f, auc_p = sent_auc(devL_items, m), prefix_auc(devL_items, m)
    for k in ['end', '4', '2']:
        sc = f'{m}|{k}'
        thr100 = calibrate(calL_items, sc, 100)
        r = evaluate(devL_items, sc, thr100)
        thr10 = calibrate_flag(calL_items, sc, 0.10)
        r10 = evaluate(devL_items, sc, thr10)
        res.append({'model': m, 'kind': k, 'recall_ARL100': r['recall'], 'delay_med_ARL100': r['delay_tok_median'],
                    'delay_chars_ARL100': r['delay_chars_median'], 'realized_ARL0': r['realized_ARL0_dev'],
                    'clean_flag_ARL100': r['clean_resp_flagged'], 'pre_onset_ARL100': r['pre_onset_alarm_rate'],
                    'recall_svc10': r10['recall'], 'delay_med_svc10': r10['delay_tok_median'],
                    'clean_flag_svc10': r10['clean_resp_flagged'], 'auc_full': auc_f, 'auc_prefix': auc_p,
                    'thr100': thr100, 'thr10': thr10})
devL_res = pd.DataFrame(res)
print(devL_res.drop(columns=['thr100', 'thr10']).round(3).to_string(index=False))

seeds = [m for m in SC if m.startswith('S')]
if seeds:
    agg = devL_res[devL_res.model.isin(seeds)].groupby('kind')[
        ['recall_ARL100', 'delay_med_ARL100', 'recall_svc10', 'delay_med_svc10', 'auc_full', 'auc_prefix']].agg(['mean', 'std'])
    print('\n시드 평균 ± 표준편차 (dev-L)'); print(agg.round(3).to_string())

## 11. 쌍체 부트스트랩 — dev-L에서 구간이 얼마나 좁아졌나

임계값은 고정하고 환각 응답만 재표집(1,000회). 같은 계산을 기존 dev(검증 보정)에도 해서 구간 폭을 비교한다.

In [ ]:
def per_resp(items, sc, thr):
    out = []
    for x in items:
        if x['onset'] is None: continue
        first = next((t for t, s in series(x, sc) if s >= thr), None)
        ok = first is not None and first >= x['onset']
        out.append((ok, first - x['onset'] if ok else np.nan))
    return np.array([o[0] for o in out], dtype=float), np.array([o[1] for o in out])

def boot(items, cal_items, a, b, n=1000, seed=0):
    da, la = per_resp(items, a, calibrate(cal_items, a, 100))
    db, lb = per_resp(items, b, calibrate(cal_items, b, 100))
    rng = np.random.default_rng(seed); N = len(da)
    rd, dd, ra = [], [], []
    for _ in range(n):
        i = rng.integers(0, N, N)
        rd.append(da[i].mean() - db[i].mean()); ra.append(da[i].mean())
        dd.append(np.nanmedian(la[i]) - np.nanmedian(lb[i]))
    q = lambda v: np.nanpercentile(v, [2.5, 97.5])
    return {'A': a, 'B': b, 'n_hall': N, 'recall_A': da.mean(), 'recall_B': db.mean(),
            'recall_diff': da.mean() - db.mean(), 'recall_diff_lo': q(rd)[0], 'recall_diff_hi': q(rd)[1],
            'recall_A_lo': q(ra)[0], 'recall_A_hi': q(ra)[1],
            'delay_diff': np.nanmedian(la) - np.nanmedian(lb), 'delay_diff_lo': q(dd)[0], 'delay_diff_hi': q(dd)[1]}

pairs = [('E2|4', 'B|end'), ('E2|4', 'B|4')] + [(f'{s}|4', 'B|end') for s in seeds] + [(f'{s}|4', 'B|4') for s in seeds]
bt = [dict(boot(devL_items, calL_items, a, b), set='dev-L') for a, b in pairs]
bt += [dict(boot(dev_items, val_items, a, b), set='dev (기존)') for a, b in pairs[:2]]
boot_df = pd.DataFrame(bt)
print(boot_df[['set', 'A', 'B', 'n_hall', 'recall_A', 'recall_A_lo', 'recall_A_hi', 'recall_diff', 'recall_diff_lo',
               'recall_diff_hi', 'delay_diff', 'delay_diff_lo', 'delay_diff_hi']].round(3).to_string(index=False))
w = boot_df.assign(width=boot_df.recall_A_hi - boot_df.recall_A_lo).groupby('set').width.mean()
print('\nE2 s=4 재현율 95% 구간 폭:', w.round(3).to_dict())

## 12. 판정 요약과 저장

드라이브 `selfproject/r0/`에 결과 표와, R2·R3 실험에 쓸 **판정 지점·점수 궤적**을 저장한다.

In [ ]:
import pickle, shutil
print('=== R0 사전 기준 ===')
print('R0-a 분할 무결성: 충족 (assert 통과)')
print('R0-b 재현        :', '충족' if R0B else '미달')
print('R0-c 오염        :', '오염 효과 있음' if R0C else '측정 범위 안에서 작음',
      f"(B 과제별 차이 평균 {contam.loc['B'].strat_diff:+.3f}, 95% [{contam.loc['B'].ci_lo:+.3f}, {contam.loc['B'].ci_hi:+.3f}])")
print('결정            : 이후 주 평가 세트 = dev-L, 보정·규칙 학습 = cal-L')

devL_res.to_csv(os.path.join(R0_DIR, 'r0_devL.csv'), index=False)
contam.to_csv(os.path.join(R0_DIR, 'r0_contamination.csv'))
contam_task.to_csv(os.path.join(R0_DIR, 'r0_contamination_task.csv'), index=False)
boot_df.to_csv(os.path.join(R0_DIR, 'r0_bootstrap.csv'), index=False)
pd.DataFrame([{'check': k, 'ref': REF[k], 'got': got[k]} for k in REF]).to_csv(os.path.join(R0_DIR, 'r0_repro.csv'), index=False)

KEEP = ['id', 'task_type', 'gen_model', 'ref_key', 'n_tok', 'onset', 'sents', 'sent_of', 'ends', 'pts2', 'ts', 'te', 'seen']
pickle.dump({'items': {k: [{f: x[f] for f in KEEP} for x in v] for k, v in ALL.items()},
             'q_keys': Q_KEYS, 'q_texts': Q_TEXTS}, open(os.path.join(R0_DIR, 'r0_items.pkl'), 'wb'))
np.savez(os.path.join(R0_DIR, 'r0_scores.npz'), **SC)
for f in ['r0_devL.csv', 'r0_contamination.csv', 'r0_contamination_task.csv', 'r0_bootstrap.csv', 'r0_repro.csv', 'r0_splits.json']:
    shutil.copy(os.path.join(R0_DIR, f), f)
os.system('zip -q r0_results.zip r0_devL.csv r0_contamination.csv r0_contamination_task.csv r0_bootstrap.csv r0_repro.csv')
print('저장 완료: 드라이브 selfproject/r0/ (r0_items.pkl, r0_scores.npz 포함), 업로드용 r0_results.zip')
try:
    from google.colab import files; files.download('r0_results.zip')
except Exception:
    pass